See how many areas/parks have been completed

In [219]:
import pandas as pd
import geopandas as gpd
import os

In [220]:
output_dir = "../workflow_outputs/"
filenames_file = "england_filenames.csv"
park_id_file = "all_parks_ids.csv"

In [221]:
filenames = pd.read_csv(filenames_file)
park_ids = pd.read_csv(park_id_file)

In [222]:
# want a subset of the parks that are in process, so we can check the progress of the workflow
first_park_index = 29
last_park_index = first_park_index + 30
filenames= filenames.iloc[first_park_index:last_park_index]

In [223]:
filenames

,filename
29,Luton_pp_or_g_cmb.geojson
30,Southend-on-Sea_pp_or_g_cmb.geojson
31,Thurrock_pp_or_g_cmb.geojson
32,Medway_pp_or_g_cmb.geojson
33,Bracknell Forest_pp_or_g_cmb.geojson
34,West Berkshire_pp_or_g_cmb.geojson
35,Reading_pp_or_g_cmb.geojson
36,Slough_pp_or_g_cmb.geojson
37,Windsor and Maidenhead_pp_or_g_cmb.geojson
38,Wokingham_pp_or_g_cmb.geojson


In [224]:
# for each filename in filenames
# get foldername by removing the .geojson
# check if the folder exists in output_dir
# add existing folder to a list

in_process_areas = []
for filename in filenames["filename"]:
    foldername = filename.replace(".geojson", "")
    folderpath = output_dir + foldername
    if os.path.exists(folderpath):
        in_process_areas.append(foldername)

In [225]:
100*len(in_process_areas)/len(filenames)

63.333333333333336

In [226]:
in_process_area_names = []
for filename in in_process_areas:
    areaname = filename.replace("_pp_or_g_cmb", "")
    in_process_area_names.append(areaname)

In [227]:
area_df = pd.DataFrame({"area_name": in_process_area_names,
                        "filename": in_process_areas,
                        "folder_path": [output_dir + filename for filename in in_process_areas]})

In [228]:
park_ids

,country,region_id,authority_id,auth_name_e,old_park_id,new_park_id
0,Wales,W10000009,W06000011,Swansea,SWANS_841e2e988f55;SWANS_9add957b9b60,3816d9f14391
1,Wales,W10000009,W06000011,Swansea,SWANS_64180ad89ee2;SWANS_7875d4152bbb,eeb8a45abba6
2,Wales,W10000009,W06000011,Swansea,SWANS_0077e0499336;SWANS_003142c8af8d,2afa4425d6cc
3,Wales,W10000009,W06000011,Swansea,SWANS_8c5a5b68c7e1;SWANS_fb7bfbb4dae0,ee93e1ee473b
4,Wales,W10000009,W06000011,Swansea,SWANS_8119996c470b,a7f47d7bf887
...,...,...,...,...,...,...
217614,England,E12000002,E08000013,St. Helens,STHEL_6a530cb0f057,e97074068e56
217615,England,E12000002,E08000013,St. Helens,STHEL_2df5277ef0c9,f9627d5e096b
217616,England,E12000002,E08000013,St. Helens,STHEL_04f0db6f2403,7957ab4ecde8
217617,England,E12000002,E08000013,St. Helens,STHEL_0384f41414f0,42c0619d2802


In [229]:
area_df

,area_name,filename,folder_path
0,Luton,Luton_pp_or_g_cmb,../workflow_outputs/Luton_pp_or_g_cmb
1,Southend-on-Sea,Southend-on-Sea_pp_or_g_cmb,../workflow_outputs/Southend-on-Sea_pp_or_g_cmb
2,Thurrock,Thurrock_pp_or_g_cmb,../workflow_outputs/Thurrock_pp_or_g_cmb
3,Medway,Medway_pp_or_g_cmb,../workflow_outputs/Medway_pp_or_g_cmb
4,Bracknell Forest,Bracknell Forest_pp_or_g_cmb,../workflow_outputs/Bracknell Forest_pp_or_g_cmb
5,West Berkshire,West Berkshire_pp_or_g_cmb,../workflow_outputs/West Berkshire_pp_or_g_cmb
6,Reading,Reading_pp_or_g_cmb,../workflow_outputs/Reading_pp_or_g_cmb
7,Slough,Slough_pp_or_g_cmb,../workflow_outputs/Slough_pp_or_g_cmb
8,Windsor and Maidenhead,Windsor and Maidenhead_pp_or_g_cmb,../workflow_outputs/Windsor and Maidenhead_pp_...
9,Wokingham,Wokingham_pp_or_g_cmb,../workflow_outputs/Wokingham_pp_or_g_cmb


In [230]:
# for each folder_path in the area_df, find all parks in the area
# in the park_ids dataframe, find all parks with area_name in the area_df
# add the total number of parks in the area to the area_df
area_df["num_parks_total"] = area_df["area_name"].apply(lambda x: len(park_ids[park_ids["auth_name_e"] == x]))
area_df["expected_park_files"] = area_df["num_parks_total"].apply(lambda x: x * 2)

In [231]:
# then count the number of parks in the associated folder_path and add that to the area_df
def count_files_in_folder(folder_path):
    park_files = [f for f in os.listdir(folder_path) if f.endswith(".geojson")]
    return len(park_files)

area_df["completed_files"] = area_df["folder_path"].apply(count_files_in_folder)
area_df["completed_parks"] = area_df["completed_files"] / 2

In [232]:
area_df

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks
0,Luton,Luton_pp_or_g_cmb,../workflow_outputs/Luton_pp_or_g_cmb,179,358,358,179.0
1,Southend-on-Sea,Southend-on-Sea_pp_or_g_cmb,../workflow_outputs/Southend-on-Sea_pp_or_g_cmb,71,142,142,71.0
2,Thurrock,Thurrock_pp_or_g_cmb,../workflow_outputs/Thurrock_pp_or_g_cmb,1703,3406,3406,1703.0
3,Medway,Medway_pp_or_g_cmb,../workflow_outputs/Medway_pp_or_g_cmb,2225,4450,4450,2225.0
4,Bracknell Forest,Bracknell Forest_pp_or_g_cmb,../workflow_outputs/Bracknell Forest_pp_or_g_cmb,129,258,258,129.0
5,West Berkshire,West Berkshire_pp_or_g_cmb,../workflow_outputs/West Berkshire_pp_or_g_cmb,175,350,350,175.0
6,Reading,Reading_pp_or_g_cmb,../workflow_outputs/Reading_pp_or_g_cmb,80,160,160,80.0
7,Slough,Slough_pp_or_g_cmb,../workflow_outputs/Slough_pp_or_g_cmb,298,596,596,298.0
8,Windsor and Maidenhead,Windsor and Maidenhead_pp_or_g_cmb,../workflow_outputs/Windsor and Maidenhead_pp_...,219,438,438,219.0
9,Wokingham,Wokingham_pp_or_g_cmb,../workflow_outputs/Wokingham_pp_or_g_cmb,155,310,310,155.0


In [233]:
area_df["pct_complete"] = 100* area_df["completed_files"]/area_df["expected_park_files"]

In [234]:
area_df

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks,pct_complete
0,Luton,Luton_pp_or_g_cmb,../workflow_outputs/Luton_pp_or_g_cmb,179,358,358,179.0,100.000000
1,Southend-on-Sea,Southend-on-Sea_pp_or_g_cmb,../workflow_outputs/Southend-on-Sea_pp_or_g_cmb,71,142,142,71.0,100.000000
2,Thurrock,Thurrock_pp_or_g_cmb,../workflow_outputs/Thurrock_pp_or_g_cmb,1703,3406,3406,1703.0,100.000000
3,Medway,Medway_pp_or_g_cmb,../workflow_outputs/Medway_pp_or_g_cmb,2225,4450,4450,2225.0,100.000000
4,Bracknell Forest,Bracknell Forest_pp_or_g_cmb,../workflow_outputs/Bracknell Forest_pp_or_g_cmb,129,258,258,129.0,100.000000
5,West Berkshire,West Berkshire_pp_or_g_cmb,../workflow_outputs/West Berkshire_pp_or_g_cmb,175,350,350,175.0,100.000000
6,Reading,Reading_pp_or_g_cmb,../workflow_outputs/Reading_pp_or_g_cmb,80,160,160,80.0,100.000000
7,Slough,Slough_pp_or_g_cmb,../workflow_outputs/Slough_pp_or_g_cmb,298,596,596,298.0,100.000000
8,Windsor and Maidenhead,Windsor and Maidenhead_pp_or_g_cmb,../workflow_outputs/Windsor and Maidenhead_pp_...,219,438,438,219.0,100.000000
9,Wokingham,Wokingham_pp_or_g_cmb,../workflow_outputs/Wokingham_pp_or_g_cmb,155,310,310,155.0,100.000000
